In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.cluster import MiniBatchKMeans
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import InstanceHardnessThreshold
from sklearn.svm import SVC
from sklearn.linear_model import SGDClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
import xgboost as xgb
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.ensemble import IsolationForest
from imblearn.ensemble import BalancedBaggingClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, accuracy_score, average_precision_score
)

In [3]:
df = pd.read_csv(r"C:\Users\ankit\OneDrive\Desktop\Research_Papers\UPI_2024_data.zip")
df.info()                     
df.describe()                 
df.nunique()                   
df.isnull().sum()              

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

transaction id        0
timestamp             0
transaction type      0
merchant_category     0
amount (INR)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64

In [4]:
df['transaction_status'] = df['transaction_status'].map({'SUCCESS': 1, 'FAILED': 0})

age_order = ['18-25', '26-35', '36-45', '46-55', '56+']
df['sender_age_group'] = pd.Categorical(df['sender_age_group'], categories=age_order, ordered=True).codes
df['receiver_age_group'] = pd.Categorical(df['receiver_age_group'], categories=age_order, ordered=True).codes

df['hour_sin'] = np.sin(2*np.pi*df['hour_of_day']/24)
df['hour_cos'] = np.cos(2*np.pi*df['hour_of_day']/24)
day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4, 'Saturday':5, 'Sunday':6}
df['day_num'] = df['day_of_week'].map(day_map)
df['day_sin'] = np.sin(2*np.pi*df['day_num']/7)
df['day_cos'] = np.cos(2*np.pi*df['day_num']/7)

df['timestamp'] = pd.to_datetime(df['timestamp'])
df['month'] = df['timestamp'].dt.month
df['day'] = df['timestamp'].dt.day
df['year'] = df['timestamp'].dt.year
df['quarter'] = df['timestamp'].dt.quarter
df['week_of_year'] = df['timestamp'].dt.isocalendar().week.astype(int)
df['month_sin'] = np.sin(2*np.pi*df['month']/12)
df['month_cos'] = np.cos(2*np.pi*df['month']/12)

df['transaction_number'] = df['transaction id'].str.extract(r'(\d+)').astype(int)

nominal_cols = ['transaction type', 'merchant_category', 'sender_state', 'sender_bank', 'receiver_bank', 'device_type', 'network_type']
df = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

df = df.drop(columns=['transaction id', 'timestamp', 'day_of_week']) 

bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print(df.shape)
print(df.select_dtypes(include='object').columns)
df.dtypes

(250000, 60)
Index([], dtype='object')


amount (INR)                         int64
transaction_status                   int64
sender_age_group                      int8
receiver_age_group                    int8
fraud_flag                           int64
hour_of_day                          int64
is_weekend                           int64
hour_sin                           float64
hour_cos                           float64
day_num                              int64
day_sin                            float64
day_cos                            float64
month                                int32
day                                  int32
year                                 int32
quarter                              int32
week_of_year                         int64
month_sin                          float64
month_cos                          float64
transaction_number                   int64
transaction type_P2M                 int64
transaction type_P2P                 int64
transaction type_Recharge            int64
merchant_ca

In [5]:
X = df.drop(columns=['fraud_flag'])
y = df['fraud_flag']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train set fraud cases:", y_train.sum(), "/", len(y_train))
print("Test set fraud cases:", y_test.sum(), "/", len(y_test), "\n")

Train set fraud cases: 384 / 200000
Test set fraud cases: 96 / 50000 



In [7]:
importance_model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
)
importance_model.fit(X_train, y_train)

importances = pd.Series(
    importance_model.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

print("Top 10 feature importances:")
print(importances.head(10), "\n")

threshold = 0.5 * importances.max()
selected_features = importances[importances >= threshold].index.tolist()

print("Selected features (>= 50% of top feature's importance):", selected_features, "\n")

X_train_full = X_train.copy()
X_train_full['fraud_flag'] = y_train.values

majority = X_train_full[X_train_full['fraud_flag'] == 0]
minority = X_train_full[X_train_full['fraud_flag'] == 1]

mask = pd.Series(True, index=majority.index)
for col in selected_features:
    Q1 = majority[col].quantile(0.25)
    Q3 = majority[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    mask &= majority[col].between(lower_bound, upper_bound)

majority_clean = majority[mask]

print("Majority class before IQR filtering:", len(majority))
print("Majority class after IQR filtering:", len(majority_clean), "\n")

cleaned = pd.concat([majority_clean, minority], axis=0)
X_clean = cleaned.drop(columns=['fraud_flag'])
y_clean = cleaned['fraud_flag']

print("Class balance after cleaning:", y_clean.value_counts().to_dict(), "\n")

sm = SMOTE(k_neighbors=3, random_state=42)
X_train_final, y_train_final = sm.fit_resample(X_clean, y_clean)

print("Class balance after SMOTE:", pd.Series(y_train_final).value_counts().to_dict())

Top 10 feature importances:
amount (INR)          0.088193
transaction_number    0.086185
day                   0.066131
week_of_year          0.053191
hour_of_day           0.047999
hour_sin              0.043473
hour_cos              0.043182
sender_age_group      0.033275
receiver_age_group    0.030465
day_sin               0.029184
dtype: float64 

Selected features (>= 50% of top feature's importance): ['amount (INR)', 'transaction_number', 'day', 'week_of_year', 'hour_of_day'] 

Majority class before IQR filtering: 199616
Majority class after IQR filtering: 182729 

Class balance after cleaning: {0: 182729, 1: 384} 

Class balance after SMOTE: {0: 182729, 1: 182729}


In [13]:
results = []

def evaluate_model(name, clf, X_train, y_train, X_test, y_test, results):
    if isinstance(clf, IsolationForest):
        clf.fit(X_train)
        raw_pred = clf.predict(X_test)
        y_pred = np.where(raw_pred == -1, 1, 0)       
        scores = -clf.decision_function(X_test)            
        probs = (scores - scores.min()) / (scores.max() - scores.min())  # normalize to [0,1]
    else:
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        probs = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else None

    pr_auc = average_precision_score(y_test, probs) if probs is not None else np.nan

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision (fraud)": precision_score(y_test, y_pred, zero_division=0),
        "Recall (fraud)": recall_score(y_test, y_pred, zero_division=0),
        "F1 (fraud)": f1_score(y_test, y_pred, zero_division=0),
        "PR-AUC": pr_auc
    })

evaluate_model("Decision Tree", 
                DecisionTreeClassifier(
                max_depth=8, 
                random_state=42),
                X_train_final, y_train_final, X_test, y_test, results)

evaluate_model("Random Forest", 
                RandomForestClassifier(
                n_estimators=300,
                max_depth=8, 
                class_weight="balanced", 
                random_state=42, 
                n_jobs=-1),
                X_train_final, y_train_final, X_test, y_test, results)

# evaluate_model("Balanced Bagging", 
#                 BalancedBaggingClassifier(
#                 estimator=DecisionTreeClassifier(max_depth=2), 
#                 random_state=42, 
#                 n_jobs=-1),
#                 X_train, y_train, X_test, y_test, results)  

evaluate_model("XGBoost", 
                xgb.XGBClassifier(
                random_state=42, 
                max_depth=2, 
                min_child_weight=5, 
                scale_pos_weight=400,
                n_estimators=150, 
                learning_rate=0.05, 
                eval_metric="logloss"),
                X_train_final, y_train_final, X_test, y_test, results)

# evaluate_model("Isolation Forest",
#                IsolationForest(
#                 n_estimators=200,
#                 max_samples=256,
#                 contamination=0.05,
#                 random_state=42,
#                 n_jobs=-1),
#                 X_train, y_train, X_test, y_test, results)

# evaluate_model("Balanced RF",
#                BalancedRandomForestClassifier(
#                 n_estimators=150,
#                 max_depth=2,          
#                 sampling_strategy='all', 
#                 replacement=True,  
#                 random_state=42,
#                 n_jobs=-1),            
#                 X_train, y_train, X_test, y_test, results)

In [14]:
results_df = pd.DataFrame(results).sort_values("F1 (fraud)", ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))

        Model  Accuracy  Precision (fraud)  Recall (fraud)  F1 (fraud)  PR-AUC
Random Forest    0.9821             0.0025          0.0208      0.0044  0.0019
      XGBoost    0.2510             0.0019          0.7396      0.0038  0.0020
Decision Tree    0.8650             0.0012          0.0833      0.0024  0.0018


In [6]:
importance_model = RandomForestClassifier(
    n_estimators=200, random_state=42, class_weight='balanced', n_jobs=-1
)
importance_model.fit(X_train, y_train)

importances = pd.Series(
    importance_model.feature_importances_, index=X_train.columns
).sort_values(ascending=False)

print("Top 10 feature importances:")
print(importances.head(10), "\n")

Top 10 feature importances:
amount (INR)          0.088193
transaction_number    0.086185
day                   0.066131
week_of_year          0.053191
hour_of_day           0.047999
hour_sin              0.043473
hour_cos              0.043182
sender_age_group      0.033275
receiver_age_group    0.030465
day_sin               0.029184
dtype: float64 

